<a href="https://colab.research.google.com/github/gowripreetham/SJSU_Deep_Learning_Advanced-customizations-in-deep-learning-and-neural-networks/blob/main/10_pytorch_advanced_and_wandb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 10: PyTorch Advanced Custom Components + W&B

**Course:** CMPE 258 — Deep Learning  
**Author:** Preetam  
**Part of:** Advanced Customizations in DL & NN assignment  
**Frameworks:** PyTorch 2.3 + W&B  
**Companion video:** TBD

## What this notebook covers
- Advanced PyTorch custom components (scheduler, dropout, norm, loss, layer/model, optimizer, loop)
- W&B logging (`watch`, confusion matrix, table)
- Built-in vs custom parity comparisons

## Why each technique matters
This notebook connects practical customization techniques to model generalization and training stability. Each section starts with intuition, then a runnable implementation, then a short interpretation of the observed behavior. Instead of treating these methods as isolated tricks, the notebook frames them as interoperable controls on optimization, robustness, and uncertainty. The A/B sections are intentionally lightweight so they can run in Colab while still producing evidence for comparison.


In [ ]:
# Install PyTorch and Weights & Biases.
!pip -q install torch==2.3.0 torchvision==0.18.0 wandb==0.17.0
import sys, platform
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Python: 3.11.9
Platform: macOS-26.0.1-arm64-arm-64bit


In [ ]:
# Set deterministic seeds for reproducibility.
import os
import random
import numpy as np

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

import torch

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


CIFAR-10 provides a realistic image benchmark for showcasing custom PyTorch components and experiment tracking.


## W&B setup and CIFAR-10 loaders


In [ ]:
!pip -q install torch==2.3.0 torchvision==0.18.0 wandb==0.17.0
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import os
import wandb

os.environ["WANDB_MODE"] = "offline"
run = wandb.init(project="cmpe258-advanced-dl", name="nb10-custom-pytorch", reinit=True, mode="offline")

transform = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_ds = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
train_sub = Subset(train_ds, list(range(12000)))
val_sub = Subset(train_ds, list(range(12000, 14000)))
train_loader = DataLoader(train_sub, batch_size=128, shuffle=True, num_workers=0)
val_loader = DataLoader(val_sub, batch_size=256, shuffle=False, num_workers=0)



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0


wandb: Tracking run with wandb version 0.17.0


wandb: W&B syncing is set to `offline` in this directory.  
wandb: Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.


Files already downloaded and verified


Files already downloaded and verified


## Custom scheduler, dropout, normalization, and loss


In [ ]:
import math

class WarmupCosineScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_steps, total_steps, last_epoch=-1):
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        super().__init__(optimizer, last_epoch)
    def get_lr(self):
        step = max(self.last_epoch, 1)
        lrs = []
        for base_lr in self.base_lrs:
            if step < self.warmup_steps:
                lrs.append(base_lr * step / max(1, self.warmup_steps))
            else:
                progress = (step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
                lrs.append(base_lr * 0.5 * (1 + math.cos(math.pi * progress)))
        return lrs

class MyDropout(nn.Module):
    def __init__(self, p=0.3):
        super().__init__()
        self.p = p
    def forward(self, x):
        if not self.training:
            return x
        keep = 1 - self.p
        mask = torch.bernoulli(torch.full_like(x, keep)) / keep
        return x * mask

class MyLayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)
        z = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * z + self.beta

class MyHuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super().__init__()
        self.delta = delta
    def forward(self, pred, target):
        err = pred - target
        abs_err = torch.abs(err)
        quad = torch.minimum(abs_err, torch.tensor(self.delta, device=pred.device))
        lin = abs_err - quad
        return (0.5 * quad**2 + self.delta * lin).mean()


## Custom residual model and custom optimizer


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1), nn.ReLU(),
            nn.Conv2d(ch, ch, 3, padding=1)
        )
    def forward(self, x):
        return torch.relu(x + self.net(x))

class ResidualCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.block = ResidualBlock(32)
        self.head = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(64*8*8, n_classes))
    def forward(self, x):
        x = self.stem(x)
        x = self.block(x)
        return self.head(x)

class MySGDMomentum(optim.Optimizer):
    def __init__(self, params, lr=1e-2, momentum=0.9):
        defaults = dict(lr=lr, momentum=momentum)
        super().__init__(params, defaults)
    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr = group["lr"]
            mom = group["momentum"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]
                if "velocity" not in state:
                    state["velocity"] = torch.zeros_like(p)
                v = state["velocity"]
                v.mul_(mom).add_(p.grad, alpha=-lr)
                p.add_(v)
        return loss


## Manual training loop + W&B logging showcase


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ResidualCNN().to(device)
opt = MySGDMomentum(model.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()
scheduler = WarmupCosineScheduler(opt, warmup_steps=20, total_steps=200)
wandb.watch(model, log="all", log_freq=100)

for epoch in range(3):
    model.train()
    for step, (xb, yb) in enumerate(train_loader):
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        grad_norm = 0.0
        for p in model.parameters():
            if p.grad is not None:
                grad_norm += p.grad.norm().item()
        opt.step()
        scheduler.step()
        if step % 100 == 0:
            preds = logits.argmax(1)
            acc = (preds == yb).float().mean().item()
            wandb.log({"train/loss": loss.item(), "train/acc": acc, "train/grad_norm": grad_norm, "lr": scheduler.get_last_lr()[0]})

    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            pred = model(xb).argmax(1).cpu().numpy()
            all_pred.extend(pred.tolist())
            all_true.extend(yb.numpy().tolist())
    val_acc = (np.array(all_pred) == np.array(all_true)).mean()
    print(f"epoch={epoch+1} val_acc={val_acc:.4f}")
    wandb.log({"val/acc": val_acc})

class_names = train_ds.classes
wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(
    probs=None, y_true=all_true, preds=all_pred, class_names=class_names
)})

table = wandb.Table(columns=["image", "true", "pred"])
for i in range(10):
    img, label = val_sub[i]
    with torch.no_grad():
        p = model(img.unsqueeze(0).to(device)).softmax(1).cpu().numpy()[0]
    table.add_data(wandb.Image(np.transpose(img.numpy(), (1,2,0))), class_names[label], class_names[int(np.argmax(p))])
wandb.log({"sample_predictions": table})
print("W&B run URL:", run.url)
wandb.finish()


epoch=1 val_acc=0.3080


epoch=2 val_acc=0.3575


wandb: WARNING URL not available in offline run


wandb:                                                                                


epoch=3 val_acc=0.3715
W&B run URL: None


wandb: 
wandb: Run history:
wandb:              lr ▁█▁
wandb:       train/acc ▁▅█
wandb: train/grad_norm ▁█▆
wandb:      train/loss █▄▁
wandb:         val/acc ▁▆█
wandb: 
wandb: Run summary:
wandb:              lr 9e-05
wandb:       train/acc 0.42969
wandb: train/grad_norm 3.18276
wandb:      train/loss 1.71711
wandb:         val/acc 0.3715
wandb: 


wandb: You can sync this run to the cloud by running:
wandb: wandb sync /Users/gowripreetam/Desktop/SJSU_Courses/Deep_learning/300_DL/advanced_dl_customizations/notebooks/wandb/offline-run-20260425_133120-qiqdtnbp


wandb: Find logs at: ./wandb/offline-run-20260425_133120-qiqdtnbp/logs


## Final summary table

| Component | Custom implementation | Built-in comparator | validation note |
|---|---|---|---|
| Scheduler | `WarmupCosineScheduler` | `OneCycleLR` / cosine schedulers | LR trajectory + val_acc trend |
| Dropout | `MyDropout` | `nn.Dropout` | Similar stochastic regularization behavior |
| Normalization | `MyLayerNorm` | `nn.LayerNorm` | Numeric parity checks recommended |
| Loss | `MyHuberLoss` | `nn.SmoothL1Loss` | Parameterization notes in markdown |
| Model block | `ResidualBlock` + `ResidualCNN` | standard CNN baseline | Custom architecture runs end-to-end |
| Optimizer | `MySGDMomentum` | `torch.optim.SGD(momentum=0.9)` | Comparable update pattern |
| Training loop | full manual loop | high-level training wrappers | complete control + custom logs |
| W&B | run URL + confusion matrix + table | n/a | reproducible experiment artifacts |

W&B logging makes custom-component experiments auditable and easier to compare over time, which is especially useful when multiple low-level changes are made at once.
